# 03 - Modeling (Baseline)

En este notebook entrenamos un modelo baseline de regresion para predecir `SalePrice` y calculamos metricas de evaluacion.

Metricas usadas:
- RMSE
- MAE
- R2

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [ ]:
# Carga de datos y seleccion de variables base

DATA_PATH = Path("../data/train.csv")
df = pd.read_csv(DATA_PATH)

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF",
    "FullBath",
    "YearBuilt",
]
target = "SalePrice"

X = df[features].copy()
y = df[target].copy()

print("Shape X:", X.shape)
print("Shape y:", y.shape)

In [ ]:
# Split train/test

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Test:", X_test.shape, y_test.shape)

In [ ]:
# Modelo baseline: RandomForest + imputacion mediana

baseline_model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1,
            ),
        ),
    ]
)

baseline_model.fit(X_train, y_train)
y_pred = baseline_model.predict(X_test)

In [ ]:
# Metricas de evaluacion

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")
print(f"R2: {r2:.4f}")

In [ ]:
# Tabla resumen para documentacion

baseline_results = pd.DataFrame(
    [
        {
            "version": "baseline_random_forest",
            "rmse": round(rmse, 4),
            "mae": round(mae, 4),
            "r2": round(r2, 4),
        }
    ]
)

baseline_results

In [ ]:
# Guardar resultados para reutilizar en informe y app

OUT_PATH = Path("../artifacts/metrics/model_comparison.csv")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
baseline_results.to_csv(OUT_PATH, index=False)

print(f"Metricas guardadas en: {OUT_PATH}")

## Interpretacion rapida

- Un RMSE y MAE menores indican mejor precision.
- Un R2 mas cercano a 1 indica mayor capacidad explicativa.
- Este notebook define la linea base que luego se comparara con el modelo tuneado en `04_tuning.ipynb`.